# BiPert: perturbation transfer with a low-rank linear operator

Companion notebook of *How Far Can Linear Go? BiPert, a Unified Framework for Single-Cell Perturbation Modelling*.

This notebook recomputes, from a compact data bundle, every number the paper reports for the
**perturbation-transfer** task (Section 4.1 for the models, Section 5 for the protocol and results).

**The task in one paragraph.** A *knockdown* switches one gene off in a population of cells. Its *effect* is a
vector with one entry per measured gene: the mean expression of the cells carrying the knockdown minus the mean
expression of untouched *control* cells. Effects are known for every knockdown in some *source* cell lines. In
a new cell line they are known for only part of the knockdowns, and the model has to predict the rest. It is
matrix completion with vector-valued entries.

**What this notebook computes**

1. Accuracy on the four-line benchmark (K562, RPE1, HepG2, Jurkat): source average, COMPASS-X and BiPert (our
   low-rank operator) at rank 50.
2. The same three methods on the harder Orion pair (HCT116, HEK293T).
3. The "code *r*" columns: how well the knockdown-specific part of the effect is transferred.
4. The rank ablation: accuracy and discrimination (PDS) of the operator as its rank goes from 10 to 300.
5. The statement that a JIVE-style split into shared and line-specific components gives nearly the same results as a
   plain SVD.
6. The few-shot table: 10, 25, 50 or 100 knockdowns observed in the new line.

Rows of the paper's tables marked with a dagger are numbers published by others (COMPASS, Table 3). They are
shown here as quoted constants, labelled **published, not recomputed**.

**Requirements:** numpy, scipy, pandas, matplotlib. **Runtime:** about 25 minutes on 2 CPU cores (a free Colab
instance), of which 15 are the JIVE comparison of Section 5.
Setting `FAST = True` in the configuration cell runs a clearly labelled subset in a few minutes.

## Configuration

In [ ]:
from pathlib import Path

DATA_DIR = Path("./transfer_data")
# Google Colab: copy the transfer_data folder to a folder BiPert on your Drive, then uncomment the next two lines.
# from google.colab import drive; drive.mount("/content/drive")
# DATA_DIR = Path("/content/drive/MyDrive/BiPert/transfer_data")

FAST = False           # True: a quick subset (2 splits, 4 halvings, 2 JIVE ranks). NOT the paper's protocol:
                       # numbers then differ from the paper by split-to-split noise (about 0.01).

N_SPLITS = 5           # random 80/20 splits of the knockdowns of the held-out line (seeds 0-4)
N_HALVINGS = 20        # random halvings of the held-out line's control cells (see "de-biased Pearson")
OPERATOR_RANK = 50     # rank of BiPert in the main tables
RANKS = [10, 30, 50, 70, 100, 150, 200, 300]   # rank ablation
JIVE_RANKS = [1, 2, 5, 10, 20, 50]             # ranks at which JIVE and the plain SVD are compared
N_OBSERVED = [10, 25, 50, 100]                 # few-shot table
FEWSHOT_RANK, FEWSHOT_K = 30, 30               # few-shot table: operator rank and basis size of the map

if FAST:
    N_SPLITS, N_HALVINGS, JIVE_RANKS = 2, 4, [10, 50]
PROTOCOL = "FAST SUBSET, not the paper's protocol" if FAST else "full protocol"

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.linalg import eigh

pd.set_option("display.width", 200)
T_START = time.time()

## Numbers published by others

`PUBLISHED` holds the dagger rows of the transfer table: values reported by the COMPASS authors on the same
benchmark, which we quote and do **not** recompute.

In [ ]:
# Published by others (COMPASS, Table 3), de-biased Pearson: (four lines, Orion pair). Not recomputed here.
PUBLISHED = {"GEARS": (0.24, 0.05), "State": (0.27, 0.09), "scGPT": (0.32, 0.11), "GenePert": (0.39, 0.18),
             "COMPASS-N": (0.40, 0.21), "COMPASS-H": (0.41, 0.19),
             "TabICL (external gene features)": (0.48, 0.24)}

## Data

The bundle has one file per cell line. Each holds, on a panel of 2,000 genes:

* `pert_mean`: one row per knockdown, the mean expression profile of the cells carrying it;
* `ctrl`: one row per control cell;
* `perts`, `genes`, `symbols`: knockdown names and gene names.

Values are log(1 + x) of counts normalised to 10,000 per cell. The **effect** of knockdown *p* in line *l* is
`pert_mean[p] - ctrl.mean(0)` (Eq. 1 of the paper).

For the metric below we also need the mean of the control cells over random halves of the control population.
The halvings are drawn here once per line (random generator seeded with 7), and the control cells are then
released to keep memory low.

In [ ]:
class Benchmark:
    """Knockdown mean profiles, control means and control half-means for a set of cell lines."""

    def __init__(self, lines):
        self.lines = lines
        self.pert_mean, self.ctrl_mean, self.ctrl_halves = {}, {}, {}
        for c in lines:
            z = np.load(DATA_DIR / f"{c}.npz")
            if c == lines[0]:
                self.perts, self.symbols = z["perts"], z["symbols"]
            assert (z["perts"] == self.perts).all() and (z["symbols"] == self.symbols).all()
            self.pert_mean[c] = z["pert_mean"].astype(np.float64)
            ctrl = z["ctrl"].astype(np.float64)
            self.ctrl_mean[c] = ctrl.mean(0)
            rng = np.random.default_rng(7)
            orders = [rng.permutation(len(ctrl)) for _ in range(N_HALVINGS)]
            self.ctrl_halves[c] = [(ctrl[o[: len(o) // 2]].mean(0), ctrl[o[len(o) // 2:]].mean(0)) for o in orders]
            print(f"{c}: {self.pert_mean[c].shape[0]} knockdowns x {self.pert_mean[c].shape[1]} genes, {len(ctrl)} control cells")
        # effect of every knockdown in every line (Eq. 1)
        self.effect = {c: self.pert_mean[c] - self.ctrl_mean[c] for c in lines}
        # column of the knocked-down gene itself in the panel, -1 if it is not one of the 2,000 genes
        self.own_gene = pd.Index(self.symbols).get_indexer(self.perts)
        self.n = len(self.perts)


four = Benchmark(["K562", "RPE1", "HepG2", "Jurkat"])

## Metrics

**De-biased Pearson.** For each test knockdown we correlate, across the 2,000 genes, the predicted effect with
the measured effect, and average over test knockdowns. *De-biased* means that the measured effect is computed
against one random half of the control cells while everything the model sees from the new line is computed
against the other half; otherwise the noise of the shared control mean appears on both sides and inflates
the correlation. The score is averaged over 20 random halvings.

**PDS (perturbation discrimination score).** For each test knockdown, we rank the measured effects of all test
knockdowns by cosine similarity to its prediction and ask where the correct one lands: the score is the
fraction of the others ranked below it, minus 0.5, so 0 is chance and 0.5 is perfect. The entry of the
knocked-down gene itself is left out, as in the COMPASS benchmark, because it is trivially large.

**Code *r*.** COMPASS splits an effect into a multiple of one direction `u` shared by all knockdowns of a line
(the mean effect, normalised) and a remainder `g`, the knockdown-specific *code*. Code *r* is the Pearson
correlation, across genes, between the remainder of the prediction and the remainder of the measured effect,
averaged over test knockdowns. It asks whether a method transfers what is specific to a knockdown, not only
the response common to all of them. This one is computed with the full control mean.

In [ ]:
def pearson_rows(P, T):
    """Pearson correlation between matching rows of P and T."""
    P, T = P - P.mean(1, keepdims=True), T - T.mean(1, keepdims=True)
    return (P * T).sum(1) / np.sqrt((P ** 2).sum(1) * (T ** 2).sum(1)).clip(1e-12)


def pds(P, T, own_gene):
    """Perturbation discrimination score minus 0.5 (cosine similarity, knocked-down gene left out)."""
    dots, pn, tn = P @ T.T, (P ** 2).sum(1), np.tile((T ** 2).sum(1), (len(P), 1))
    r = np.where(own_gene >= 0)[0]
    dots[r] -= P[r, own_gene[r]][:, None] * T[:, own_gene[r]].T
    pn[r] -= P[r, own_gene[r]] ** 2
    tn[r] -= T[:, own_gene[r]].T ** 2
    cos = dots / np.sqrt(pn[:, None] * tn).clip(1e-12)
    n_better = (cos > np.diag(cos)[:, None]).sum(1)
    return float((1 - n_better / len(P)).mean() - 0.5)

## The COMPASS decomposition

Three small functions, re-implemented from the COMPASS paper: the shared direction `u` of a line, the split
of effects into a coefficient along `u` and a remainder, and the inverse of that split.

In [ ]:
def shared_axis(Y):
    """Unit vector along the mean effect of a line."""
    mean = Y.mean(0)
    return mean / max(float(np.linalg.norm(mean)), 1e-12)


def decompose(Y, u):
    """Effects -> (coefficient along u, remainder orthogonal to u)."""
    beta = Y @ u
    return beta, Y - beta[:, None] * u[None, :]


def assemble(beta, g, u):
    return beta[:, None] * u[None, :] + g

## Methods

Every method is written the same way. It is built once from the source-line effects (a list of
knockdowns x genes arrays). `fit(tr, te)` then prepares it for one split, where `tr` are the knockdowns observed
in the new line and `te` the ones to predict. The result is a function that maps the observed effects `Y_obs`
(one row per knockdown in `tr`) to predictions for `te`. The last step is separate because it is repeated for
every halving of the control cells.

* **Source average**: the mean effect of the same knockdown in the source lines. It ignores the new line.
* **COMPASS-X**: coefficient and remainder are averaged over the source lines; only the direction `u` is
  estimated in the new line, from the observed effects.
* **BiPert (ours), the low-rank operator**: effect = `V_l U_p`. The codes `U_p` are the top-*r* left singular vectors of
  the matrix whose row *p* stacks the effects of knockdown *p* in all source lines. The matrix `V` of the new
  line is fitted by least squares on the observed knockdowns.

Two implementation details of the operator, identical to the runs behind the paper: each source line is
centred and scaled to unit Frobenius norm before the SVD, so that no line dominates, and a constant 1 is
appended to every code, so that `V` can also represent the new line's mean effect.

In [ ]:
def with_intercept(U):
    return np.hstack([np.ones((len(U), 1)), U])


def source_average(sources):
    average = np.mean(sources, 0)
    return lambda tr, te: (lambda Y_obs: average[te])


def compass_x(sources):
    parts = [decompose(Y, shared_axis(Y)) for Y in sources]
    beta = np.mean([b for b, _ in parts], 0)
    g = np.mean([g for _, g in parts], 0)
    return lambda tr, te: (lambda Y_obs: assemble(beta[te], g[te], shared_axis(Y_obs)))


def top_left_singular_vectors(X, r):
    _, vectors = eigh(X @ X.T, subset_by_index=[X.shape[0] - r, X.shape[0] - 1])
    return vectors[:, ::-1]


def normalised_blocks(sources):
    blocks = [Y - Y.mean(0) for Y in sources]
    return [B / np.linalg.norm(B) for B in blocks]


def shared_codes(sources, rank):
    """Codes U (knockdowns x rank): top left singular vectors of the stacked source effects."""
    return top_left_singular_vectors(np.hstack(normalised_blocks(sources)), rank)


def lowrank_operator(U):
    def fit(tr, te):
        # least squares is linear in the observed effects: predictions = H @ Y_obs
        H = with_intercept(U[te]) @ np.linalg.pinv(with_intercept(U[tr]))
        return lambda Y_obs: H @ Y_obs
    return fit

## Evaluation protocol

Each cell line is held out in turn; the others are the sources and contribute all their knockdowns. In the
held-out line a random 80% of the knockdowns are observed and 20% are predicted, over five splits (seeds 0-4).
For the few-shot table, only `n_observed` of the 80% are observed (drawn with seed `split + 100`); the test
knockdowns stay the same.

For every halving of the control cells, the measured test effects use the first half and the observed effects
given to the model use the second half.

In [ ]:
def split(n, seed):
    order = np.random.default_rng(seed).permutation(n)
    return np.sort(order[: int(0.8 * n)]), np.sort(order[int(0.8 * n):])


def evaluate(bench, held_out, methods, n_observed=None):
    """Score methods {name: fit} on one held-out line. One row per (split, method)."""
    rows = []
    for seed in range(N_SPLITS):
        pool, te = split(bench.n, seed)
        tr = pool if n_observed is None else np.sort(np.random.default_rng(seed + 100).permutation(pool)[:n_observed])
        predictors = {name: fit(tr, te) for name, fit in methods.items()}
        scores = {name: [] for name in methods}
        for half_1, half_2 in bench.ctrl_halves[held_out]:
            truth = bench.pert_mean[held_out][te] - half_1
            Y_obs = bench.pert_mean[held_out][tr] - half_2
            for name, predict in predictors.items():
                P = predict(Y_obs)
                scores[name].append((pearson_rows(P, truth).mean(), pds(P, truth, bench.own_gene[te])))
        # code r, with the full control mean
        u = shared_axis(bench.effect[held_out][tr])
        g_true = decompose(bench.effect[held_out][te], u)[1]
        for name, predict in predictors.items():
            g_pred = decompose(predict(bench.effect[held_out][tr]), u)[1]
            pearson, pds_score = np.mean(scores[name], 0)
            rows.append({"line": held_out, "split": seed, "method": name, "pearson": pearson, "pds": pds_score,
                         "code_r": pearson_rows(g_pred, g_true).mean()})
    return pd.DataFrame(rows)

## 1. Four-line benchmark: accuracy

**Claim.** De-biased Pearson, averaged over the four held-out lines and five splits: source average 0.32,
COMPASS-X 0.38, BiPert at rank 50 0.47 (0.465 at three decimals).

The run below also scores the operator at every rank of the ablation (Section 4), so that the four lines are
processed once.

**How to read it.** Higher is better; 1 would be a perfect prediction of every test effect. The second table
gives the per-line values behind the mean.

In [ ]:
def four_line_methods(sources):
    U = shared_codes(sources, max(RANKS))            # top singular vectors are nested: rank r = first r columns
    methods = {"Source average": source_average(sources), "COMPASS-X": compass_x(sources)}
    methods.update({f"rank {r}": lowrank_operator(U[:, :r]) for r in RANKS})
    return methods


runs = []
for held_out in four.lines:
    sources = [four.effect[c] for c in four.lines if c != held_out]
    runs.append(evaluate(four, held_out, four_line_methods(sources)))
    print(f"{held_out} done ({time.time() - T_START:.0f} s since start)")
res4 = pd.concat(runs, ignore_index=True)
mean4 = res4.groupby("method")[["pearson", "pds", "code_r"]].mean()
OPERATOR = f"rank {OPERATOR_RANK}"

In [ ]:
METHODS = ["Source average", "COMPASS-X", OPERATOR]
print("Four lines, de-biased Pearson (mean over splits):")
print(mean4.loc[METHODS, "pearson"].round(3).to_string())
print("\nPer held-out line:")
print(res4[res4.method.isin(METHODS)]
      .pivot_table(index="method", columns="line", values="pearson")[four.lines].round(3).to_string())
per_split = res4.groupby(["method", "split"]).pearson.mean().groupby("method").std()
print(f"\ns.d. across splits of the four-line mean: COMPASS-X {per_split['COMPASS-X']:.3f}, operator {per_split[OPERATOR]:.3f}")

## 2. Orion pair: accuracy

**Claim.** On the pair HCT116 / HEK293T (2,270 knockdowns, each line held out with the other as the single
source): source average 0.08, COMPASS-X 0.12, BiPert at rank 50 0.25.

**How to read it.** Same metric as Section 1. All numbers are lower because one source line carries less
information than three; the question is the gap between methods. With a single source line, the "source
average" is simply that line's effect, and the codes are the top singular vectors of that line's effects.

In [ ]:
orion = Benchmark(["HCT116", "HEK293T"])
runs = []
for held_out in orion.lines:
    sources = [orion.effect[c] for c in orion.lines if c != held_out]
    runs.append(evaluate(orion, held_out, {
        "Source average": source_average(sources), "COMPASS-X": compass_x(sources),
        OPERATOR: lowrank_operator(shared_codes(sources, OPERATOR_RANK))}))
resO = pd.concat(runs, ignore_index=True)
meanO = resO.groupby("method")[["pearson", "pds", "code_r"]].mean()

In [ ]:
print("Orion pair, de-biased Pearson (mean over splits):")
print(meanO.loc[METHODS, "pearson"].round(3).to_string())
print("\nPer held-out line:")
print(resO.pivot_table(index="method", columns="line", values="pearson")[orion.lines].round(3).to_string())

### The transfer table and figure, with the published rows

The table below is the paper's transfer table (de-biased Pearson columns). Rows labelled *published, not
recomputed* are constants quoted from COMPASS's Table 3; the other rows come from the two runs above.

In [ ]:
recomputed = {"Source average": "Source average", "COMPASS-X": "COMPASS-X", "BiPert (ours)": OPERATOR}
table = pd.DataFrame(
    [{"method": m, "four lines": a, "Orion pair": b, "source": "published, not recomputed"} for m, (a, b) in PUBLISHED.items()]
    + [{"method": m, "four lines": mean4.loc[k, "pearson"], "Orion pair": meanO.loc[k, "pearson"], "source": "recomputed here"}
       for m, k in recomputed.items()])
print(table.to_string(index=False, float_format=lambda v: f"{v:.2f}"))

GREY, BLUE = "#9aa0a6", "#2a78d6"
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), sharey=True)
for ax, column in zip(axes, ["four lines", "Orion pair"]):
    colours = [GREY if s.startswith("published") else BLUE for s in table["source"]]
    bars = ax.barh(table["method"], table[column], color=colours, height=0.7)
    ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=9)
    ax.set_title(column, fontsize=10, loc="left")
    ax.set_xlabel("de-biased Pearson")
    ax.set_xlim(0, table[column].max() * 1.18)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(length=0)
axes[0].invert_yaxis()                               # the y axis is shared: first table row on top in both panels
fig.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=GREY), plt.Rectangle((0, 0), 1, 1, color=BLUE)],
           labels=["published, not recomputed", "recomputed here"], frameon=False, fontsize=9, ncol=2,
           loc="lower center", bbox_to_anchor=(0.5, -0.07))
fig.suptitle("Perturbation transfer: mean over held-out lines and splits", fontsize=11, x=0.01, ha="left")
fig.tight_layout()
plt.show()

## 3. Transfer of the knockdown-specific code ("code *r*")

**Claim.** Code *r* on the four lines: COMPASS-X 0.22, BiPert 0.34. On the Orion pair: source
average 0.06, COMPASS-X 0.07 (0.065 at three decimals), BiPert 0.20.

**How to read it.** This is the correlation between predicted and measured effects after the direction shared
by all knockdowns has been projected out (see "Metrics"). A method that only predicts the common response
scores 0 here.

In [ ]:
print("Transfer of the knockdown-specific code (code r):")
print(pd.DataFrame({"four lines": mean4.loc[METHODS, "code_r"],
                    "Orion pair": meanO.loc[METHODS, "code_r"]}).round(3).to_string())

## 4. Rank ablation: accuracy against discrimination

**Claims.** (a) Accuracy rises from 0.43 at rank 10 to a plateau of 0.47 at ranks 50-100 (0.465 at rank 50) and falls to 0.44 at
rank 300. (b) PDS rises steadily with rank, from 0.15 at rank 10 (0.145 at three decimals) to 0.25 at rank 300. (c) At rank 50 the
operator's PDS is below COMPASS-X's (0.21 against 0.25), and it matches COMPASS-X only at ranks 200-300.

**How to read it.** The rank is the only setting of the operator. A low rank averages out noise and gives a
better correlation, but it also discards the directions that tell similar knockdowns apart, which is what PDS
measures.

In [ ]:
by_rank = mean4.loc[[f"rank {r}" for r in RANKS]].set_axis(RANKS).rename_axis("rank")
print(by_rank.round(3).to_string())
print(f"COMPASS-X: pearson {mean4.loc['COMPASS-X', 'pearson']:.3f}, pds {mean4.loc['COMPASS-X', 'pds']:.3f}\n")


## 5. A JIVE-style split changes almost nothing

**Claim.** Splitting the factorisation into components shared by all source lines and components specific to
each line (JIVE, Lock et al. 2013) changes the results of the plain SVD by at most 0.004 in Pearson and 0.010 in PDS across ranks.

**What is computed.** JIVE alternates between (i) an SVD of the stacked source lines after removing each
line's individual part, which gives the shared codes, and (ii) an SVD of what the shared codes leave
unexplained in each line, which gives that line's individual part (rank 10 here). The shared codes replace
the plain-SVD codes; the rest of the operator is unchanged. Both versions are scored at ranks 1 to 50.

**How to read it.** The table lists JIVE minus plain SVD for both metrics, and the cell below prints the
largest absolute gap over the ranks.

In [ ]:
def jive_codes(sources, rank, individual_rank=10, iterations=10, tol=1e-3):
    blocks = normalised_blocks(sources)
    individual = [np.zeros_like(B) for B in blocks]
    previous = None
    for _ in range(iterations):
        S = top_left_singular_vectors(np.hstack([B - A for B, A in zip(blocks, individual)]), rank)
        for i, B in enumerate(blocks):
            R = B - S @ (S.T @ (B - individual[i]))     # the line minus its shared part
            R -= S @ (S.T @ R)                          # individual part is orthogonal to the shared codes
            W = top_left_singular_vectors(R, individual_rank)
            individual[i] = W @ (W.T @ R)
        if previous is not None and 1 - np.linalg.norm(previous.T @ S) ** 2 / rank < tol:
            break
        previous = S
    return S


runs = []
for held_out in four.lines:
    sources = [four.effect[c] for c in four.lines if c != held_out]
    methods = {("JIVE", r): lowrank_operator(jive_codes(sources, r)) for r in JIVE_RANKS}
    methods.update({("plain SVD", r): lowrank_operator(shared_codes(sources, r)) for r in JIVE_RANKS})
    runs.append(evaluate(four, held_out, methods))
    print(f"{held_out} done ({time.time() - T_START:.0f} s since start)")
resJ = pd.concat(runs, ignore_index=True)
resJ[["codes", "rank"]] = pd.DataFrame(resJ["method"].tolist(), index=resJ.index)
meanJ = resJ.pivot_table(index="rank", columns="codes", values=["pearson", "pds"])
gap = (meanJ.xs("JIVE", axis=1, level="codes") - meanJ.xs("plain SVD", axis=1, level="codes")).add_prefix("JIVE - SVD: ")
print(pd.concat([meanJ.round(4), gap.round(4)], axis=1).to_string())

In [ ]:
print("JIVE compared with the plain SVD, largest gap over ranks:")
print(f"Pearson {gap['JIVE - SVD: pearson'].abs().max():.4f}, PDS {gap['JIVE - SVD: pds'].abs().max():.4f}")

## 6. Few-shot: 10 to 100 knockdowns observed in the new line

**Claim.** The full few-shot table: de-biased Pearson on the four lines when only 10, 25, 50 or 100 knockdowns
are observed in the held-out line, for the source average, COMPASS-X, BiPert (rank 30) and
BiPert-FS (basis size 30).

**The two models of ours in this table.**

* **BiPert, few-shot version.** With so few observations, the matrix `V` of the new line is not
  fitted from scratch: it is ridge-regressed (strength 100) towards the average of the source lines' own
  matrices. With zero observations it would return the rank-30 approximation of the source average.
* **BiPert-FS** (Eq. 7 of the paper). Take the source-average effect, express it in a basis `Phi`
  of 30 gene-space directions (top right singular vectors of the source effects), apply a learned affine map
  `b + C a` to those 30 coordinates, and leave the part outside the basis unchanged. `b` and `C` are fitted by
  ridge regression on the observed knockdowns, with `C` pulled towards the identity. The ridge strength is
  `alpha * n_observed * (mean variance of the source coordinates)`, and `alpha` is chosen among
  {0.01, 0.1, 1, 10} **on the source lines only**: each source line plays the new line against the other two,
  and the `alpha` with the best Pearson + PDS wins.

**How to read it.** Each column is a budget of observed knockdowns; the test knockdowns are the same at every
budget. The source average does not use the new line, so its row is constant.

In [ ]:
ALPHAS = [0.01, 0.1, 1.0, 10.0]


def lowrank_operator_fewshot(sources, rank, ridge=100.0):
    n = len(sources[0])
    means = [Y.mean(0) for Y in sources]
    blocks = [Y - m for Y, m in zip(sources, means)]
    U = top_left_singular_vectors(np.hstack([B / np.linalg.norm(B) for B in blocks]), rank) * np.sqrt(n)
    V_sources = np.mean([U.T @ B / n for B in blocks], 0)      # average of the source lines' matrices
    mean_sources = np.mean(means, 0)

    def prior(idx):
        return mean_sources + U[idx] @ V_sources

    def fit(tr, te):
        A = with_intercept(U[tr])
        H = with_intercept(U[te]) @ np.linalg.solve(A.T @ A + ridge * np.eye(rank + 1), A.T)
        return lambda Y_obs: prior(te) + H @ (Y_obs - prior(tr))
    return fit


def basis_and_average(sources, k):
    """Phi (genes x k, orthonormal columns) and the source-average effect."""
    Phi = np.linalg.svd(np.concatenate([Y - Y.mean(0) for Y in sources]), full_matrices=False)[2][:k].T
    return Phi, np.mean(sources, 0)


def fewshot_map(Phi, average, alpha):
    k = Phi.shape[1]
    a_src = average @ Phi                                       # source coordinates of every knockdown

    def fit(tr, te):
        mu = alpha * len(tr) * np.mean(np.var(a_src, axis=0))
        X = with_intercept(a_src[tr])
        penalty = np.zeros((k + 1, k + 1))
        penalty[1:, 1:] = mu * np.eye(k)                        # b is not penalised
        target = np.vstack([np.zeros((1, k)), np.eye(k)])       # C is pulled towards the identity

        def predict(Y_obs):
            W = np.linalg.solve(X.T @ X + penalty, X.T @ (Y_obs @ Phi) + penalty @ target)
            return with_intercept(a_src[te]) @ W @ Phi.T + average[te] - a_src[te] @ Phi.T
        return predict
    return fit


def choose_alphas(bench, source_lines, k):
    """alpha per budget, chosen with each source line as pseudo-target against the other sources."""
    score = {b: np.zeros(len(ALPHAS)) for b in N_OBSERVED}
    pool, te = split(bench.n, 1000)
    order = np.random.default_rng(1100).permutation(pool)
    for pseudo in source_lines:
        Phi, average = basis_and_average([bench.effect[c] for c in source_lines if c != pseudo], k)
        truth = bench.effect[pseudo][te]
        for b in N_OBSERVED:
            tr = np.sort(order[:b])
            for j, alpha in enumerate(ALPHAS):
                P = fewshot_map(Phi, average, alpha)(tr, te)(bench.effect[pseudo][tr])
                score[b][j] += pearson_rows(P, truth).mean() + pds(P, truth, bench.own_gene[te])
    return {b: ALPHAS[int(score[b].argmax())] for b in N_OBSERVED}

In [ ]:
runs = []
for held_out in four.lines:
    source_lines = [c for c in four.lines if c != held_out]
    sources = [four.effect[c] for c in source_lines]
    alphas = choose_alphas(four, source_lines, FEWSHOT_K)
    Phi, average = basis_and_average(sources, FEWSHOT_K)
    fixed = {"Source average": source_average(sources), "COMPASS-X": compass_x(sources),
             "BiPert": lowrank_operator_fewshot(sources, FEWSHOT_RANK)}
    for b in N_OBSERVED:
        methods = {**fixed, "BiPert-FS": fewshot_map(Phi, average, alphas[b])}
        runs.append(evaluate(four, held_out, methods, n_observed=b).assign(n_observed=b))
    print(f"{held_out} done; alpha chosen on the sources per budget: {alphas} ({time.time() - T_START:.0f} s since start)")
resF = pd.concat(runs, ignore_index=True)
ORDER = ["Source average", "COMPASS-X", "BiPert", "BiPert-FS"]
meanF = resF.pivot_table(index="method", columns="n_observed", values="pearson").loc[ORDER]
print("\nFew-shot table (de-biased Pearson):")
print(meanF.round(3).to_string())
sd = resF.groupby(["method", "n_observed", "split"]).pearson.mean().groupby(["method", "n_observed"]).std().unstack()
print("\ns.d. across the 5 splits of the four-line mean:")
print(sd.loc[ORDER].round(3).to_string())